In [1]:
# PySpark Imports
import pyspark
from pyspark.sql import SparkSession

# ML Classifier Imports
from pyspark.ml.classification import LinearSVC
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import VectorAssembler, StringIndexer, PCA
from pyspark.ml.classification import OneVsRest
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.sql.functions import mean, col
import time
import os
import sys

In [2]:
# Initialize Spark session
spark = SparkSession.builder.appName("ce53") \
    .master("local[*]") \
    .config("spark.driver.cores", "2") \
    .config("spark.driver.memory", "14g") \
    .config("spark.executor.memory", "14g") \
    .config("spark.executor.cores", "2") \
    .config("spark.dynamicAllocation.shuffleTracking.enabled", "true") \
    .config("spark.dynamicAllocation.enabled", "true") \
    .config("spark.dynamicAllocation.minExecutors", "2") \
    .config("spark.dynamicAllocation.maxExecutors", "2") \
    .config("spark.executor.instances", "2") \
    .config("spark.kryoserializer.buffer.max", "2047m") \
    .config("spark.sql.execution.pythonUDF.arrow.enabled", "false") \
    .config("spark.executor.heartbeatInterval","11999s") \
    .config("spark.network.timeout","12000s") \
.getOrCreate()

24/04/09 20:00:13 WARN Utils: Your hostname, colin-MS-7977 resolves to a loopback address: 127.0.1.1; using 192.168.0.164 instead (on interface wlx3c52a1d3ccda)
24/04/09 20:00:13 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/04/09 20:00:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/04/09 20:00:15 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
parquet_files = ["Parquet/part-00000-1da06990-329c-4e38-913a-0f0aa39b388d-c000.snappy.parquet", "Parquet/part-00000-26e9208e-7819-451b-b23f-2e47f6d1e834-c000.snappy.parquet", 
                 "Parquet/part-00000-36240b61-b84f-4164-a873-d7973e652780-c000.snappy.parquet", "Parquet/part-00000-3f86626a-1225-47f9-a5a2-0170b737e404-c000.snappy.parquet",
                 "Parquet/part-00000-7c2e9adb-5430-4792-a42b-10ff5bbd46e8-c000.snappy.parquet", "Parquet/part-00000-b1a9fc13-8068-4a5d-91b2-871438709e81-c000.snappy.parquet",
                 "Parquet/part-00000-cbf26680-106d-40e7-8278-60520afdbb0e-c000.snappy.parquet", "Parquet/part-00000-df678a79-4a73-452b-8e72-d624b2732f17-c000.snappy.parquet"]

In [4]:
# Read the parquet files into a dataframe
df = spark.read.parquet(*parquet_files, inferSchema=True)

In [5]:
# Get unique labels and their counts
label_counts = df.groupBy("label_tactic").count().orderBy("label_tactic")

# Show the results
label_counts.show()

+--------------------+-------+
|        label_tactic|  count|
+--------------------+-------+
|   Credential Access|     31|
|     Defense Evasion|      1|
|           Discovery|   2086|
|        Exfiltration|      7|
|      Initial Access|      1|
|    Lateral Movement|      4|
|         Persistence|      1|
|Privilege Escalation|     13|
|      Reconnaissance|9278722|
|Resource Development|      3|
|                none|9281599|
+--------------------+-------+



In [6]:
start_time = time.time()

# List of labels to drop
labels_to_drop = ["Defense Evasion", "Exfiltration", "Initial Access", "Lateral Movement", "Persistence", "Privilege Escalation", "Resource Development", "Credential Access", "Reconnaissance"]

# Filter out the rows with labels to drop
df = df.filter(~col("label_tactic").isin(labels_to_drop))

# Get unique labels and their counts after filtering
filtered_label_counts = df.groupBy("label_tactic").count().orderBy("label_tactic")

# Show the filtered results
filtered_label_counts.show()


end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

+------------+-------+
|label_tactic|  count|
+------------+-------+
|   Discovery|   2086|
|        none|9281599|
+------------+-------+

Execution time: 0.8682081699371338 seconds


In [7]:
start_time = time.time()



df = df.withColumn("datetime", col("datetime").cast("string"))

# Define columns to index
columns_to_index = ['service', 'conn_state', 'history', 'proto', 'dest_ip_zeek', 'community_id', 'uid', 'src_ip_zeek', 'label_tactic', 'datetime']

# Impute null values with 'null' string
for column in columns_to_index:
    df = df.fillna('null', subset=[column])

# Apply StringIndexer to each column
indexers = [StringIndexer(inputCol=column, outputCol=column+"_indexed").fit(df) for column in columns_to_index]

# Chain indexers together
pipeline = Pipeline(stages=indexers)

# Fit and transform the data
df_indexed = pipeline.fit(df).transform(df)

# Drop original columns
df_indexed = df_indexed.drop(*columns_to_index)

# Show the schema of the DataFrame
#df_indexed.show()



end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

Execution time: 48.309139013290405 seconds


In [8]:
# Split the data into training and test sets
start_time = time.time()

train_data, test_data = df_indexed.randomSplit([0.7, 0.3], seed=42)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

Execution time: 0.01814126968383789 seconds


In [9]:
from pyspark.ml.feature import Imputer


start_time = time.time()

# List of numeric column names
numeric_columns = ['resp_pkts', 'orig_ip_bytes', 'missed_bytes', 'duration', 'orig_pkts',
                   'resp_ip_bytes', 'dest_port_zeek', 'orig_bytes', 'resp_bytes',
                   'src_port_zeek', 'ts']


# Create an Imputer object
imputer = Imputer(
    inputCols=numeric_columns,
    outputCols=["{}_imputed".format(column) for column in numeric_columns]
)

# Fit the imputer to the training data
imputer_model = imputer.setStrategy("mean").fit(train_data)

# Apply the imputer to the training data
train_data_imputed = imputer_model.transform(train_data)


end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")


start_time = time.time()


# Apply the imputer to the test data
test_data_imputed = imputer_model.transform(test_data)

# Show updated DataFrames
#train_data_imputed.show()
#test_data_imputed.show()



end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

24/04/09 20:01:23 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB


Execution time: 40.205153703689575 seconds
Execution time: 0.01865673065185547 seconds


In [10]:
from pyspark.ml.feature import VectorAssembler


start_time = time.time()



# List of columns to assemble
columns_to_assemble = [column for column in train_data_imputed.columns if column.endswith("_imputed")]

# Create the VectorAssembler
assembler = VectorAssembler(inputCols=columns_to_assemble, outputCol="features")

# Transform the training DataFrame
train_data_assembled = assembler.transform(train_data_imputed)


end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")


start_time = time.time()
# Transform the test DataFrame
test_data_assembled = assembler.transform(test_data_imputed)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")



# Select only the features and label columns for both training and test sets
train_data_assembled = train_data_assembled.select("features", "label_tactic_indexed")
test_data_assembled = test_data_assembled.select("features", "label_tactic_indexed")

# Show the schema of the assembled training DataFrame
#train_data_assembled.printSchema()

# Show the schema of the assembled test DataFrame
#test_data_assembled.printSchema()

Execution time: 5.824603080749512 seconds
Execution time: 0.08117008209228516 seconds


In [11]:
# Create the SVM model
start_time = time.time()
svm = LinearSVC(labelCol="label_tactic_indexed", featuresCol="features", maxIter=10, regParam=0.0, tol=.00001, fitIntercept=True)

# One Vs. Rest
ovr = OneVsRest(classifier=svm, labelCol='label_tactic_indexed')

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

# Fit the model
start_time = time.time()

svm_model = ovr.fit(train_data_assembled)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

Execution time: 0.021041154861450195 seconds


24/04/09 20:02:07 WARN DAGScheduler: Broadcasting large task binary with size 265.9 MiB
24/04/09 20:02:49 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB
24/04/09 20:03:32 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB
24/04/09 20:03:52 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB
24/04/09 20:03:57 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
24/04/09 20:04:08 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB
24/04/09 20:04:28 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB
24/04/09 20:04:45 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB
24/04/09 20:05:01 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB
24/04/09 20:05:17 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB
24/04/09 20:05:36 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB
24/04/09 20:0

Execution time: 5508.283742427826 seconds


In [12]:
# Make predictions
start_time = time.time()

predictions = svm_model.transform(test_data_assembled)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

Execution time: 0.38674163818359375 seconds


In [13]:
# Evaluate the model
# Calculate accuracy
start_time = time.time()
evaluator_accuracy = MulticlassClassificationEvaluator(labelCol="label_tactic_indexed", metricName="accuracy")
accuracy = evaluator_accuracy.evaluate(predictions)
print("Accuracy:", accuracy)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

# Calculate precision
start_time = time.time()
evaluator_precision = MulticlassClassificationEvaluator(labelCol="label_tactic_indexed", metricName="weightedPrecision")
precision = evaluator_precision.evaluate(predictions)
print("Precision:", precision)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

# Calculate recall
start_time = time.time()
evaluator_recall = MulticlassClassificationEvaluator(labelCol="label_tactic_indexed", metricName="weightedRecall")
recall = evaluator_recall.evaluate(predictions)
print("Recall:", recall)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

# Calculate F1-score
start_time = time.time()
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label_tactic_indexed", metricName="f1")
f1_score = evaluator_f1.evaluate(predictions)
print("F1-Score:", f1_score)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")



print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-Score:", f1_score)

24/04/09 21:34:16 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB


Accuracy: 1.0
Execution time: 84.51693534851074 seconds


24/04/09 21:35:37 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB


Precision: 1.0
Execution time: 79.1762146949768 seconds


24/04/09 21:36:59 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB


Recall: 1.0
Execution time: 85.58593821525574 seconds


24/04/09 21:38:26 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB


F1-Score: 1.0
Execution time: 99.2362151145935 seconds
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1-Score: 1.0


In [14]:
from pyspark.sql.functions import expr

start_time = time.time()

# Extract Predictions and True Labels
predictions_and_labels = predictions.select("prediction", "label_tactic_indexed")

# Calculate False Positives
false_positives = predictions_and_labels.filter((predictions_and_labels.prediction == 1) & (predictions_and_labels.label_tactic_indexed == 0)).count()

# Calculate True Negatives
true_negatives = predictions_and_labels.filter((predictions_and_labels.prediction == 0) & (predictions_and_labels.label_tactic_indexed == 0)).count()

# Calculate False Positive Rate (FPR)
fpr = false_positives / (false_positives + true_negatives)

print("False Positive Rate:", fpr)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")


24/04/09 21:40:02 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB
24/04/09 21:41:31 WARN DAGScheduler: Broadcasting large task binary with size 266.0 MiB


False Positive Rate: 0.0
Execution time: 184.11059498786926 seconds


In [15]:
spark.sparkContext.stop()